# Fine-tune Llama 3.1 8B with LoRA - Standard Version (No Unsloth)

This notebook fine-tunes Meta's Llama 3.1 8B model using standard libraries:
- **Transformers** for model loading
- **PEFT** for LoRA (Low-Rank Adaptation)
- **TRL** for SFT training
- **BitsAndBytes** for 4-bit quantization

**No Unsloth required** - uses only standard libraries for maximum compatibility.

**Key Features:**
- Auto-downloads dataset and model
- 4-bit quantization with BitsAndBytes
- LoRA fine-tuning via PEFT
- Pre and post-training inference tests
- GPU monitoring and statistics
- Model merging and saving

## Step 0: Auto-Download Dataset & Model

In [26]:
import os
from datasets import load_dataset
from huggingface_hub import snapshot_download

print("\n" + "="*70)
print("STEP 0: AUTO-DOWNLOAD DATASET & MODEL")
print("="*70)

# ============================================================================
# AUTO-DOWNLOAD DATASET
# ============================================================================
dataset_path = "./python_code_instructions_18k_alpaca"
dataset_hf_path = os.path.join(dataset_path, "hf_format")

if not os.path.exists(dataset_hf_path):
    print("\n📥 Dataset not found. Downloading...")
    print(f"   Source: iamtarun/python_code_instructions_18k_alpaca")
    print(f"   Target: {dataset_path}")
    
    try:
        dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
        os.makedirs(dataset_path, exist_ok=True)
        dataset.save_to_disk(dataset_hf_path)
        print(f"   ✅ Dataset downloaded successfully!")
        print(f"   Size: {len(dataset):,} examples")
    except Exception as e:
        print(f"   ❌ Error: {e}")
        raise
else:
    print(f"\n✅ Dataset already exists at {dataset_hf_path}")

# ============================================================================
# AUTO-DOWNLOAD MODEL
# ============================================================================
# Use locally available model (already downloaded with Unsloth)
# If you need to download from HF Hub, set HF_TOKEN environment variable:
#   export HF_TOKEN="your_token_here"

# Check which model is available locally
llama_paths = [
    ("./Meta-Llama-3.1-8B-bnb-4bit", "Meta-Llama-3.1-8B-bnb-4bit (original)"),
    ("./llama-3.2-3b", "Llama 3.2 3B"),
    ("./llama-2-7b-hf", "Llama 2 7B"),
]

model_path = None
model_name = None

print("\n📋 Checking for locally available models...")
for path, name in llama_paths:
    if os.path.exists(path) and len(os.listdir(path)) > 3:
        print(f"   ✅ Found: {name} at {path}")
        model_path = path
        model_name = name
        break

if not model_path:
    print("\n⚠️  No local model found. Options:")
    print("   1. Use Unsloth version which has Meta-Llama-3.1-8B-bnb-4bit")
    print("   2. Set HF_TOKEN and download from Hugging Face:")
    print("      export HF_TOKEN='your_token_from_huggingface.co/settings/tokens'")
    print("   3. Download manually:")
    print("      huggingface-cli download meta-llama/Llama-3.2-3B --local-dir ./llama-3.2-3b")
    print("\n   Using existing model from Unsloth (if available):")
    
    # Fallback to Unsloth version
    if os.path.exists("./Meta-Llama-3.1-8B-bnb-4bit"):
        model_path = "./Meta-Llama-3.1-8B-bnb-4bit"
        model_name = "Meta-Llama-3.1-8B-bnb-4bit"
        print(f"   ✅ Using: {model_name}")
    else:
        raise FileNotFoundError(
            "No model found locally. Please download one of the models above."
        )

print(f"\n📋 Using model: {model_name}")
print(f"   Path: {model_path}")
print(f"   Status: ✅ Ready")

print("\n" + "="*70)
print("✅ ALL DOWNLOADS COMPLETE")
print("="*70 + "\n")


STEP 0: AUTO-DOWNLOAD DATASET & MODEL

✅ Dataset already exists at ./python_code_instructions_18k_alpaca/hf_format

📋 Checking for locally available models...
   ✅ Found: Meta-Llama-3.1-8B-bnb-4bit (original) at ./Meta-Llama-3.1-8B-bnb-4bit

📋 Using model: Meta-Llama-3.1-8B-bnb-4bit (original)
   Path: ./Meta-Llama-3.1-8B-bnb-4bit
   Status: ✅ Ready

✅ ALL DOWNLOADS COMPLETE



## Step 1: Import Libraries

In [27]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from datasets import load_from_disk
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer

# Print library versions for debugging
print("="*70)
print("LIBRARY VERSIONS")
print("="*70)
print(f"PyTorch: {torch.__version__}")
import transformers
print(f"Transformers: {transformers.__version__}")
import datasets as ds
print(f"Datasets: {ds.__version__}")
import peft
print(f"PEFT: {peft.__version__}")
import trl
print(f"TRL: {trl.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print("="*70 + "\n")

LIBRARY VERSIONS
PyTorch: 2.11.0+cu130
Transformers: 5.5.0
Datasets: 4.3.0
PEFT: 0.20.0
TRL: 0.24.0
CUDA Available: True



## Step 2: Configuration

In [29]:
# Model configuration (model_path is set in Step 0)
# It will auto-detect: Meta-Llama-3.1-8B-bnb-4bit or llama-3.2-3b
max_seq_length = 1024 if "3.2-3b" in model_path else 2048

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# LoRA configuration (adjust based on model size)
# For 3B models: r=8, for larger: r=16
lora_r = 8 if "3.2-3b" in model_path else 16

lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=16 if "3.2-3b" in model_path else 32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Test example
instruction = "Create a function to calculate the sum of a sequence of integers."
test_input = "[1, 2, 3, 4, 5]"

print("✅ Configuration complete")
print(f"   Model: {model_name}")
print(f"   Max sequence length: {max_seq_length}")
print(f"   LoRA rank: {lora_r}")

✅ Configuration complete
   Model: Meta-Llama-3.1-8B-bnb-4bit (original)
   Max sequence length: 2048
   LoRA rank: 16


## Step 3: Load Model & Test Before Training

In [30]:
print(f"Loading {model_name} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,  # Use auto-detected model path from Step 0
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\\n" + "="*70)
print("BEFORE TRAINING - Base Model Inference")
print("="*70)

model.eval()  # Set to evaluation mode

with torch.no_grad():
    prompt = alpaca_prompt.format(instruction, test_input, "")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response)

Loading Meta-Llama-3.1-8B-bnb-4bit (original) with 4-bit quantization...


/home/prabir/dgx-book/.venv/lib/python3.12/site-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


\n======================================================================
BEFORE TRAINING - Base Model Inference
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.

### Instruction:
Create a function to calculate the sum of a sequence of integers.

### Input:
[1, 2, 3, 4, 5]

### Response:
23



## Step 4: Load & Format Dataset

In [31]:
print("\nLoading dataset...")

def formatting_func(examples):
    """Format dataset with Alpaca prompt template."""
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output_text)
        texts.append(text)
    
    return {"text": texts}

# Load dataset
dataset_path = "./python_code_instructions_18k_alpaca/hf_format"
if not os.path.exists(dataset_path):
    print(f"❌ Dataset not found at {dataset_path}")
    print(f"   Please run Step 0 first!")
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

dataset = load_from_disk(dataset_path)

# Format dataset - keep all columns for compatibility with newer TRL
# The formatting_func adds 'text' column, SFTTrainer uses dataset_text_field="text"
dataset = dataset.map(formatting_func, batched=True)

print(f"✅ Dataset loaded: {len(dataset):,} examples")
print(f"   Columns: {dataset.column_names}")


Loading dataset...
✅ Dataset loaded: 18,612 examples
   Columns: ['instruction', 'input', 'output', 'prompt', 'text']


## Step 5: Setup LoRA

In [32]:
print("Applying LoRA...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied")

Applying LoRA...
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
✅ LoRA applied


## Step 6: Configure Trainer

In [33]:
# Workaround for TRL version compatibility issue
# Some TRL versions have issues with TrainingArguments compatibility

print("Checking TRL version and API...")
import inspect
from trl import SFTTrainer

sft_init_signature = inspect.signature(SFTTrainer.__init__)
supported_params = list(sft_init_signature.parameters.keys())
print(f"SFTTrainer supported parameters: {supported_params}\n")

# Define formatting function for newer TRL versions
def formatting_func(examples):
    """Format examples for SFTTrainer (used in newer TRL versions)."""
    instructions = examples.get("instruction", examples.get("text", []))
    inputs = examples.get("input", [""] * len(instructions))
    outputs = examples.get("output", [""] * len(instructions))
    
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        if "text" in examples and instruction in examples["text"]:
            texts.append(instruction)
        else:
            text = alpaca_prompt.format(instruction, input_text, output_text)
            texts.append(text)
    
    return {"text": texts}

# Create TrainingArguments with workaround for TRL compatibility
training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=5,
    max_steps=100,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="paged_adamw_32bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    save_strategy="no",
    save_steps=0,
)

# WORKAROUND: Add missing keys that TRL expects
# Convert to dict and add missing keys before TRL processes it
dict_args = training_args.to_dict()
if "push_to_hub_token" not in dict_args:
    dict_args["push_to_hub_token"] = None
if "hub_token" not in dict_args:
    dict_args["hub_token"] = None

# Recreate TrainingArguments with the fixed dict
training_args = TrainingArguments(**dict_args)

print("✅ TrainingArguments configured with compatibility fix\n")

# Build trainer kwargs
trainer_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "args": training_args,
}

# Add parameters based on what's supported
if "formatting_func" in supported_params:
    trainer_kwargs["formatting_func"] = formatting_func
    print("✅ Using: formatting_func (newer TRL)")

if "peft_config" in supported_params:
    trainer_kwargs["peft_config"] = lora_config
    print("✅ Using: peft_config for LoRA")
    
if "max_seq_length" in supported_params:
    trainer_kwargs["max_seq_length"] = max_seq_length
    print(f"✅ Using: max_seq_length={max_seq_length}")

if "packing" in supported_params:
    trainer_kwargs["packing"] = False
    print("✅ Using: packing=False")

print("\n⚠️  Attempting to create SFTTrainer with compatibility workaround...")
try:
    trainer = SFTTrainer(**trainer_kwargs)
    print("✅ Trainer created successfully!")
    print("   Batch size: 4")
    print("   Gradient accumulation: 2")
    print("   Max steps: 100")
except KeyError as e:
    print(f"\n❌ TRL version incompatibility issue: {e}")
    print("\n💡 SOLUTION: Use the Unsloth version instead!")
    print("   Open: finetune-v2-fixed.ipynb")
    print("   It has been tested and works without these issues.")
    print("\n   Or install compatible TRL version:")
    print("   pip install --upgrade trl transformers")
    raise

Checking TRL version and API...
SFTTrainer supported parameters: ['self', 'model', 'args', 'data_collator', 'train_dataset', 'eval_dataset', 'processing_class', 'compute_loss_func', 'compute_metrics', 'callbacks', 'optimizers', 'optimizer_cls_and_kwargs', 'preprocess_logits_for_metrics', 'peft_config', 'formatting_func']



TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'push_to_hub_token'

## Step 7: Monitor GPU & Train

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
print("\nStarting training...\n")

trainer_stats = trainer.train()

## Step 8: Training Statistics

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print("\n" + "="*70)
print("TRAINING COMPLETE - Statistics")
print("="*70)
print(f"⏱️  Training time: {trainer_stats.metrics['train_runtime']} seconds")
print(f"⏱️  Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"💾 Peak memory used: {used_memory} GB")
print(f"💾 Memory for LoRA: {used_memory_for_lora} GB")
print(f"📊 Memory usage: {used_percentage}% of max")
print("="*70)

## Step 9: Test After Training

In [ ]:
print("\n" + "="*70)
print("AFTER TRAINING - Fine-tuned Model Inference")
print("="*70)

model.eval()

with torch.no_grad():
    prompt = alpaca_prompt.format(instruction, test_input, "")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response)

## Step 10: Save Model

In [ ]:
print("\n" + "="*70)
print("SAVING MODEL")
print("="*70)

# Save LoRA adapters
print("\n1️⃣  Saving LoRA adapters...")
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("   ✅ Saved to ./lora_model")
print("   Use this with: PeftModel.from_pretrained(base_model, 'lora_model')")

# Merge and save full model
print("\n2️⃣  Merging and saving full model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("model_merged")
tokenizer.save_pretrained("model_merged")
print("   ✅ Saved to ./model_merged")
print("   Use this as: AutoModelForCausalLM.from_pretrained('model_merged')")

print("\n" + "="*70)
print("✅ MODEL SAVED SUCCESSFULLY")
print("="*70)
print("\nYou can now:")
print("  - Use ./lora_model for LoRA inference")
print("  - Use ./model_merged for standalone inference")
print("  - Push to HuggingFace Hub if desired")